# M3 — 5-Index FAISS Comparison (Clean, Self-Contained Version)

Compares all 5 index types requested: **IndexFlatL2** (exact kNN baseline), **IndexIVFFlat**,
**IndexPQ**, **IndexIVFPQ**, **IndexHNSWFlat**.

This version does NOT depend on an external `eval_queries_final.csv` file — that file caused
repeated bugs across earlier notebook runs (wrong/missing file silently substituting the weak
auto-generated eval set). Instead, the correct hand-labeled eval set is regenerated inline,
directly from the same filter logic already verified to work, using the actual loaded device data.
This makes the notebook fully self-contained and reproducible in any fresh environment.

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faiss-cpu"])

import numpy as np
import pandas as pd
import time
import re

df = pd.read_parquet("/kaggle/input/datasets/mrnotalent/laptop-embedding/laptop_chunks_embeddings_with_lineage.parquet")
print(df.shape)

embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
print("embedding matrix:", embeddings.shape, embeddings.dtype)

device_level = df.drop_duplicates(subset="row_uid").copy()
print("unique devices:", len(device_level))

## 2. Build the CORRECT hand-labeled eval set (inline, no external file)

30 realistic compound-criteria queries, with relevance defined by applying real filters to the
actual device data — NOT by auto-generating a query from a single device's own title (the bug
that kept resurfacing). Includes the two corrections found during earlier validation:
OLED detection uses chunk-text search (not the `display` field, which never contains "OLED"),
and SSD detection matches `SSD|PCIe|NVMe` (not just the literal word "SSD").

In [ ]:
oled_row_uids = set(df.loc[df["chunk_text"].str.contains("OLED", case=False, na=False), "row_uid"])

hand_queries_v2 = [
    {"query": "cheap laptop under $400 for basic tasks",
     "filter": lambda d: d["price_usd"] < 400},
    {"query": "gaming laptop with NVIDIA RTX under $1200",
     "filter": lambda d: d["gpu"].str.contains("RTX", case=False, na=False) & (d["price_usd"] < 1200)},
    {"query": "lightweight ultrabook under 3 lbs",
     "filter": lambda d: d["category"].str.contains("Ultrabook|Thin", case=False, na=False)},
    {"query": "laptop with 32GB RAM and 1TB SSD",
     "filter": lambda d: (d["ram_gb"] >= 32) & d["storage"].str.contains("1TB|1 TB", case=False, na=False)},
    {"query": "AMD Ryzen laptop under $600",
     "filter": lambda d: d["cpu"].str.contains("Ryzen", case=False, na=False) & (d["price_usd"] < 600)},
    {"query": "Intel Core i7 laptop with NVIDIA graphics",
     "filter": lambda d: d["cpu"].str.contains("i7", case=False, na=False) & d["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)},
    {"query": "Intel Core i5 laptop under $700",
     "filter": lambda d: d["cpu"].str.contains("i5", case=False, na=False) & (d["price_usd"] < 700)},
    {"query": "laptop with a 1TB SSD under $900",
     "filter": lambda d: d["storage"].str.contains("1TB|1 TB", case=False, na=False) & d["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False) & (d["price_usd"] < 900)},
    {"query": "touchscreen 2-in-1 convertible laptop",
     "filter": lambda d: d["display"].str.contains("Touch", case=False, na=False) & d["category"].str.contains("2-in-1|Convertible|Yoga", case=False, na=False)},
    {"query": "premium laptop over $1500 with 32GB RAM",
     "filter": lambda d: (d["price_usd"] > 1500) & (d["ram_gb"] >= 32)},
    {"query": "Dell business laptop under $1000",
     "filter": lambda d: d["title"].str.contains("Dell", case=False, na=False) & (d["price_usd"] < 1000)},
    {"query": "HP laptop with AMD Ryzen processor",
     "filter": lambda d: d["title"].str.contains("HP", case=False, na=False) & d["cpu"].str.contains("Ryzen", case=False, na=False)},
    {"query": "Lenovo laptop under $500",
     "filter": lambda d: d["title"].str.contains("Lenovo", case=False, na=False) & (d["price_usd"] < 500)},
    {"query": "ASUS laptop with OLED display",
     "filter": lambda d, _oled=oled_row_uids: d["title"].str.contains("ASUS", case=False, na=False) & d["row_uid"].isin(_oled)},
    {"query": "Acer laptop under $500",
     "filter": lambda d: d["title"].str.contains("Acer", case=False, na=False) & (d["price_usd"] < 500)},
    {"query": "17 inch gaming laptop",
     "filter": lambda d: d["display"].str.contains("17", na=False) & d["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)},
    {"query": "compact 13 inch laptop under $800",
     "filter": lambda d: d["display"].str.contains("13", na=False) & (d["price_usd"] < 800)},
    {"query": "15.6 inch Full HD budget laptop under $500",
     "filter": lambda d: d["display"].str.contains("15.6", na=False) & d["display"].str.contains("FHD|Full HD", case=False, na=False) & (d["price_usd"] < 500)},
    {"query": "workstation laptop with professional GPU",
     "filter": lambda d: d["gpu"].str.contains("Quadro|RTX A", case=False, na=False)},
    {"query": "laptop with integrated graphics only under $400",
     "filter": lambda d: d["gpu"].str.contains("Intel", case=False, na=False) & ~d["gpu"].str.contains("NVIDIA|AMD|Radeon", case=False, na=False) & (d["price_usd"] < 400)},
    {"query": "laptop with over 10 hours battery life",
     "filter": lambda d: d["battery"].notna() & d["battery"].str.contains(r"1[0-9]\s*Hour|[2-9][0-9]\s*Hour", case=False, na=False, regex=True)},
    {"query": "Chromebook under $300",
     "filter": lambda d: d["title"].str.contains("Chromebook", case=False, na=False)},
    {"query": "512GB SSD laptop under $600",
     "filter": lambda d: d["storage"].str.contains("512", na=False) & d["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False) & (d["price_usd"] < 600)},
    {"query": "Intel Core Ultra laptop with 16GB RAM",
     "filter": lambda d: d["cpu"].str.contains("Core Ultra", case=False, na=False) & (d["ram_gb"] >= 16)},
    {"query": "laptop under $300 for basic browsing",
     "filter": lambda d: d["price_usd"] < 300},
    {"query": "mid-range laptop $700-$1000 with SSD",
     "filter": lambda d: (d["price_usd"] >= 700) & (d["price_usd"] <= 1000) & d["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False)},
    {"query": "gaming laptop under $1000",
     "filter": lambda d: d["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False) & (d["price_usd"] < 1000)},
    {"query": "business laptop 14 inch under $1200",
     "filter": lambda d: d["display"].str.contains("14", na=False) & (d["price_usd"] < 1200)},
    {"query": "budget student laptop 8GB RAM",
     "filter": lambda d: (d["ram_gb"] >= 8) & (d["ram_gb"] < 16) & (d["price_usd"] < 500)},
    {"query": "high performance laptop 64GB RAM",
     "filter": lambda d: d["ram_gb"] >= 64},
]

eval_rows = []
for item in hand_queries_v2:
    mask = item["filter"](device_level)
    relevant_uids = device_level.loc[mask, "row_uid"].tolist()
    eval_rows.append({
        "query": item["query"],
        "n_relevant": len(relevant_uids),
        "relevant_row_uids": relevant_uids,
    })

eval_df_correct = pd.DataFrame(eval_rows)
print(eval_df_correct[["query", "n_relevant"]].to_string())

zero_hit = eval_df_correct[eval_df_correct["n_relevant"] == 0]
if len(zero_hit):
    print("\n\u26a0\ufe0f WARNING: queries with zero matching devices \u2014 filter logic may need adjustment for this data:")
    print(zero_hit["query"].tolist())
else:
    print("\nAll 30 queries have at least one matching device \u2014 eval set is usable.")

## 3. Encode eval queries (used for both recall@k-vs-exact and hit-rate evaluation)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # must match embedding model used in M2
query_embeddings = model.encode(eval_df_correct["query"].tolist(), batch_size=32, show_progress_bar=False).astype("float32")
print(query_embeddings.shape)

## 4. Build all 5 indices

In [ ]:
import faiss

d = embeddings.shape[1]  # 384
n = embeddings.shape[0]
build_stats = {}

def build_and_time(build_fn, name):
    t0 = time.perf_counter()
    index = build_fn()
    build_time_s = time.perf_counter() - t0
    mem_mb = len(faiss.serialize_index(index)) / (1024 ** 2)
    build_stats[name] = (build_time_s, mem_mb)
    print(f"{name:20s} | build: {build_time_s:.3f}s | memory: {mem_mb:.2f} MB")
    return index

# 1. Baseline: exact brute-force search (this IS the kNN baseline)
def _build_flat():
    idx = faiss.IndexFlatL2(d)
    idx.add(embeddings)
    return idx
index_flat = build_and_time(_build_flat, "IndexFlatL2")

# 2. IVF
def _build_ivf():
    nlist_local = min(50, n // 10)
    quantizer = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFFlat(quantizer, d, nlist_local)
    idx.train(embeddings)
    idx.add(embeddings)
    idx.nprobe = 8
    return idx
index_ivf = build_and_time(_build_ivf, "IndexIVFFlat")

# 3. HNSW
def _build_hnsw():
    idx = faiss.IndexHNSWFlat(d, 32)
    idx.hnsw.efConstruction = 40
    idx.add(embeddings)
    idx.hnsw.efSearch = 32
    return idx
index_hnsw = build_and_time(_build_hnsw, "IndexHNSWFlat")

# 4. PQ
m = 8
nbits = 8
def _build_pq():
    idx = faiss.IndexPQ(d, m, nbits)
    idx.train(embeddings)
    idx.add(embeddings)
    return idx
index_pq = build_and_time(_build_pq, "IndexPQ")

# 5. IVF+PQ
nlist = min(64, max(8, n // 10))
def _build_ivfpq():
    quantizer_pq = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFPQ(quantizer_pq, d, nlist, m, nbits)
    idx.train(embeddings)
    idx.add(embeddings)
    idx.nprobe = 8
    return idx
index_ivfpq = build_and_time(_build_ivfpq, "IndexIVFPQ")

print("\nAll 5 indices built:", index_flat.ntotal, index_ivf.ntotal, index_hnsw.ntotal, index_pq.ntotal, index_ivfpq.ntotal)

## 5. Evaluate all 5: recall@5 vs. exact search, latency, build time, memory

In [ ]:
def evaluate_index(index, name, query_vecs, k=5):
    _, gt_indices = index_flat.search(query_vecs, k)

    start = time.perf_counter()
    _, pred_indices = index.search(query_vecs, k)
    elapsed = time.perf_counter() - start
    latency_ms_per_query = (elapsed / len(query_vecs)) * 1000

    hits, total = 0, 0
    for gt_row, pred_row in zip(gt_indices, pred_indices):
        hits += len(set(gt_row.tolist()) & set(pred_row.tolist()))
        total += len(gt_row)
    recall_at_k = hits / total

    build_time_s, mem_mb = build_stats[name]
    print(f"{name:20s} | recall@{k}: {recall_at_k:.3f} | avg latency: {latency_ms_per_query:.3f} ms/query "
          f"| build: {build_time_s:.3f}s | memory: {mem_mb:.2f} MB")
    return {"index": name, "recall_at_5": recall_at_k, "latency_ms": latency_ms_per_query,
            "build_time_s": build_time_s, "memory_mb": mem_mb}

results = []
for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat"),
                   (index_pq, "IndexPQ"), (index_ivfpq, "IndexIVFPQ")]:
    results.append(evaluate_index(idx, name, query_embeddings, k=5))

results_df = pd.DataFrame(results)
results_df.to_csv("m3_index_comparison_5way.csv", index=False)
results_df

## 6. Hit rate for all 5, against the SAME correct eval set built in section 2

In [ ]:
def compute_hit_rate(index, eval_df, k=5):
    queries = eval_df["query"].tolist()
    query_vecs = model.encode(queries, batch_size=32, show_progress_bar=False).astype("float32")
    _, I = index.search(query_vecs, k)

    hits = []
    for i, row in eval_df.iterrows():
        relevant_uids = set(row["relevant_row_uids"])  # already a list, no need to split a string
        retrieved_uids = set(df.iloc[I[i]]["row_uid"].astype(str).tolist())
        hits.append(len(retrieved_uids & relevant_uids) > 0)
    return sum(hits) / len(hits)

hit_rate_results = []
for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat"),
                   (index_pq, "IndexPQ"), (index_ivfpq, "IndexIVFPQ")]:
    hr = compute_hit_rate(idx, eval_df_correct, k=5)
    hit_rate_results.append({"index": name, "hit_rate": hr})
    print(f"{name:20s} hit rate: {hr:.3f}")

hit_rate_df = pd.DataFrame(hit_rate_results)
hit_rate_df.to_csv("m3_hit_rate_5way.csv", index=False)

# merged, single source-of-truth table for the report
final_table = results_df.merge(hit_rate_df, on="index")
final_table.to_csv("m3_final_5way_comparison.csv", index=False)
print()
print(final_table.to_string(index=False))

## 7. Save outputs

In [ ]:
faiss.write_index(index_hnsw, "laptop_index_hnsw.faiss")
df[["chunk_id", "row_uid", "title", "chunk_text"]].to_csv("faiss_id_lookup.csv", index=False)
eval_df_correct.to_csv("eval_queries_final_regenerated.csv", index=False)

print("Saved: m3_index_comparison_5way.csv, m3_hit_rate_5way.csv, m3_final_5way_comparison.csv,")
print("       laptop_index_hnsw.faiss, faiss_id_lookup.csv, eval_queries_final_regenerated.csv")

try:
    from google.colab import files
    for f in ["m3_final_5way_comparison.csv", "eval_queries_final_regenerated.csv", "laptop_index_hnsw.faiss", "faiss_id_lookup.csv"]:
        files.download(f)
except ImportError:
    print("(Not in Colab \u2014 files are saved in the working directory; download manually if on Kaggle.)")

## 8. Precision@5 / Recall@5 for all 5 indices (extends Table 6.2 to full 5-way coverage)

Table 6.2 in the report previously only covered IndexFlatL2 and IndexHNSWFlat (the only two
indices that existed when that table was first built). This section extends the same
precision@5/recall@5 methodology \u2014 measured against genuine hand-labeled relevance, not
agreement with exact search \u2014 to all 5 index architectures, using the identical `eval_df_correct`
built in Section 2 so results are directly comparable across both tables.

In [ ]:
def precision_recall_at_k(index, eval_df, k=5):
    queries = eval_df["query"].tolist()
    query_vecs = model.encode(queries, batch_size=32, show_progress_bar=False).astype("float32")
    _, I = index.search(query_vecs, k)

    precisions, recalls = [], []
    for i, row in eval_df.iterrows():
        retrieved_uids = set(df.iloc[I[i]]["row_uid"].astype(str).tolist())
        relevant_uids = set(row["relevant_row_uids"])
        if len(relevant_uids) == 0:
            continue  # skip unusable queries (shouldn't occur given Section 2's zero-hit check)
        hits = len(retrieved_uids & relevant_uids)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant_uids))

    return np.mean(precisions), np.mean(recalls)


pr_results = []
for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat"),
                   (index_pq, "IndexPQ"), (index_ivfpq, "IndexIVFPQ")]:
    p, r = precision_recall_at_k(idx, eval_df_correct, k=5)
    pr_results.append({"index": name, "precision_at_5": p, "recall_at_5": r})
    print(f"{name:20s} | precision@5: {p:.3f} | recall@5: {r:.3f}")

pr_df = pd.DataFrame(pr_results)
pr_df.to_csv("m3_precision_recall_5way.csv", index=False)

# merge with the hit-rate table from Section 6 for one complete Table 6.2 source file
table_6_2_full = pr_df.merge(hit_rate_df, on="index")
table_6_2_full.to_csv("table_6_2_full_5way.csv", index=False)
print()
print(table_6_2_full.to_string(index=False))

try:
    from google.colab import files
    files.download("m3_precision_recall_5way.csv")
    files.download("table_6_2_full_5way.csv")
except ImportError:
    print("(Not in Colab \u2014 files saved in working directory.)")